# Day 11 v3 — Gold: All Blob Marts (Production — Job Parameters)

**Production notebook. Attach to ADF pipeline or Databricks Job.**

**Source:** Silver blob tables (Day 9) + Silver API dimension tables (Day 8)
**Sink:** `gold-volume/blob/` (4 Gold Delta marts)

## Parameters

| Parameter | Source | Required | Default | Notes |
|---|---|---|---|---|
| `load_type` | ADF `baseParameters` | Yes | `incremental` | `full` or `incremental` |
| `run_date` | ADF `baseParameters` | No | yesterday UTC | For realtime sessions filter (YYYY-MM-DD) |
| `run_year` | ADF `baseParameters` | No | previous UTC month's year | For invoice + report filter |
| `run_month` | ADF `baseParameters` | No | previous UTC month | For invoice + report filter (MM) |

## Gold marts

| Mart | Grain | MERGE key |
|---|---|---|
| `realtime_sessions_enriched` | 1 row/session | `session_id` |
| `invoice_summary` | 1 row/year+month | `invoice_year` + `invoice_month` |
| `kpi_sla_dashboard` | 1 row/report_period | `report_period` |
| `state_performance` | 1 row/state/report_period | `state_code` + `report_period` |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone, timedelta

print('Imports OK')

In [ ]:
# Cell 2 — Parameters
# load_type from ADF. All date params default based on data cadence:
#   run_date  → yesterday UTC (realtime sessions are hourly, process yesterday's batch)
#   run_year/month → previous UTC month (invoices + reports are monthly)

_now        = datetime.now(timezone.utc)
_yesterday  = _now - timedelta(days=1)
_prev_month = _now.replace(day=1) - timedelta(days=1)  # last day of prior month

def _get_param(key, default):
    try:
        val = dbutils.widgets.get(key).strip()
        return val if val else default
    except Exception:
        return default

LOAD_TYPE = _get_param('load_type', 'incremental')
RUN_DATE  = _get_param('run_date',  _yesterday.strftime('%Y-%m-%d'))
RUN_YEAR  = _get_param('run_year',  str(_prev_month.year))
RUN_MONTH = _get_param('run_month', f'{_prev_month.month:02d}')

if LOAD_TYPE not in ('full', 'incremental'):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f'load_type : {LOAD_TYPE}')
print(f'run_date  : {RUN_DATE}  (realtime filter)')
print(f'run_year  : {RUN_YEAR}')
print(f'run_month : {RUN_MONTH}  (invoice + report filter)')
print('Parameters resolved.')

In [ ]:
# Cell 3 — Constants
SILVER_REALTIME = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
SILVER_INVOICES = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/invoices'
SILVER_REPORTS  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/reports'
SILVER_API      = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
GOLD_BLOB       = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/blob'
PIPELINE        = 'pl_gold_blob_all_marts_v3'
RUN_TS          = _now.strftime('%Y-%m-%dT%H:%M:%SZ')

print(f'Gold blob : {GOLD_BLOB}')
print(f'RUN_TS    : {RUN_TS}')

In [ ]:
# Cell 4 — Helper functions
def folder_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def write_gold(df, mart_name, merge_keys, partition_cols=None):
    path = f'{GOLD_BLOB}/{mart_name}'
    if LOAD_TYPE == 'full' or not folder_exists(path):
        writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
        return 'overwrite'
    else:
        condition = ' AND '.join([f'target.{k} = source.{k}' for k in merge_keys])
        DeltaTable.forPath(spark, path).alias('target') \
            .merge(df.alias('source'), condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        return 'merge'

print('Helpers defined')

In [ ]:
# Cell 5 — Read Silver tables (with incremental filters)

# Realtime: filter to run_date for incremental
rt_sessions_all = spark.read.format('delta').load(f'{SILVER_REALTIME}/charging_sessions')
rt_sessions = (
    rt_sessions_all.filter(F.to_date('plug_in_ts') == F.lit(RUN_DATE))
    if LOAD_TYPE == 'incremental'
    else rt_sessions_all
)

# Invoices: filter to run_year/run_month for incremental
invoices_all = spark.read.format('delta').load(SILVER_INVOICES)
invoices = (
    invoices_all.filter(
        (F.col('invoice_year')  == F.lit(int(RUN_YEAR))) &
        (F.col('invoice_month') == F.lit(int(RUN_MONTH)))
    )
    if LOAD_TYPE == 'incremental'
    else invoices_all
)

# Reports: filter to run_year/run_month for incremental
REPORT_PERIOD = f'{RUN_YEAR}{RUN_MONTH}'
kpi_report_all  = spark.read.format('delta').load(f'{SILVER_REPORTS}/kpi_report')
sla_report_all  = spark.read.format('delta').load(f'{SILVER_REPORTS}/sla_report')
state_bdown_all = spark.read.format('delta').load(f'{SILVER_REPORTS}/state_breakdown')

kpi_report  = kpi_report_all.filter(F.col('report_period') == F.lit(REPORT_PERIOD))  if LOAD_TYPE == 'incremental' else kpi_report_all
sla_report  = sla_report_all.filter(F.col('report_period') == F.lit(REPORT_PERIOD))  if LOAD_TYPE == 'incremental' else sla_report_all
state_bdown = state_bdown_all.filter(F.col('report_period') == F.lit(REPORT_PERIOD)) if LOAD_TYPE == 'incremental' else state_bdown_all

# API dimensions — always full read (slowly changing, no date filter)
customers     = spark.read.format('delta').load(f'{SILVER_API}/customers')
stations      = spark.read.format('delta').load(f'{SILVER_API}/stations')
chargers      = spark.read.format('delta').load(f'{SILVER_API}/chargers')
tariffs       = spark.read.format('delta').load(f'{SILVER_API}/tariffs')
energy_prices = spark.read.format('delta').load(f'{SILVER_API}/energy_prices')
weather       = spark.read.format('delta').load(f'{SILVER_API}/weather')
cities        = spark.read.format('delta').load(f'{SILVER_API}/cities')
states_dim    = spark.read.format('delta').load(f'{SILVER_API}/states')

print(f'rt_sessions  : {rt_sessions.count()} rows  (load_type={LOAD_TYPE}, date={RUN_DATE})')
print(f'invoices     : {invoices.count()} rows  ({RUN_YEAR}/{RUN_MONTH})')
print(f'kpi_report   : {kpi_report.count()} rows  (period={REPORT_PERIOD})')
print(f'sla_report   : {sla_report.count()} rows')
print(f'state_bdown  : {state_bdown.count()} rows')

In [ ]:
# Cell 6 — Build dimension lookup tables (shared across marts)
sta_dim = stations.select(
    'station_id',
    F.col('name').alias('station_name'),
    F.col('city').alias('station_city'),
    F.col('state').alias('station_state'),
    F.col('country').alias('station_country'),
    F.col('latitude').alias('station_lat'),
    F.col('longitude').alias('station_lon'),
)
chg_dim = chargers.select(
    'charger_id', 'charger_type',
    F.col('power_kw').alias('charger_power_kw'),
    F.col('status').alias('charger_status'),
)
cust_dim = customers.select(
    'customer_id',
    F.concat_ws(' ', 'first_name', 'last_name').alias('customer_name'),
    F.col('city').alias('customer_city'),
    F.col('state').alias('customer_state'),
)
tar_dim = tariffs.select(
    'tariff_id',
    F.col('name').alias('tariff_name'),
    F.col('price_per_kwh').alias('tariff_price_per_kwh'),
    F.col('price_per_min').alias('tariff_price_per_min'),
)
ep_win = Window.partitionBy('region').orderBy(F.col('effective_from').desc())
ep_dim = (
    energy_prices
    .withColumn('_rn', F.row_number().over(ep_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select(
        F.col('region').alias('station_state'),
        F.col('price_per_kwh').alias('grid_price_per_kwh'),
    )
)
city_state = cities.select('city_id', F.col('state_code'))
weather_state = (
    weather.join(city_state, on='city_id', how='left')
    .groupBy('state_code')
    .agg(
        F.avg('temperature_c').cast('decimal(5,2)').alias('avg_temp_c'),
        F.avg('humidity_pct').cast('decimal(5,2)').alias('avg_humidity_pct'),
        F.avg('wind_speed_kmh').cast('decimal(6,2)').alias('avg_wind_kmh'),
    )
)
station_counts = (
    stations.groupBy(F.col('state').alias('state_code'))
    .agg(
        F.count('station_id').alias('total_stations'),
        F.sum('total_chargers').alias('total_chargers'),
    )
)
state_names = states_dim.select(
    'state_code',
    F.col('name').alias('state_name'),
    'country',
)

print('Dimension tables ready')

In [ ]:
# Cell 7 — Build realtime_sessions_enriched
realtime_enriched = (
    rt_sessions
    .join(sta_dim,  on='station_id',    how='left')
    .join(chg_dim,  on='charger_id',    how='left')
    .join(cust_dim, on='customer_id',   how='left')
    .join(tar_dim,  on='tariff_id',     how='left')
    .join(ep_dim,   on='station_state', how='left')
    .withColumn('session_date',  F.to_date('plug_in_ts'))
    .withColumn('session_year',  F.year('plug_in_ts'))
    .withColumn('session_month', F.month('plug_in_ts'))
    .withColumn('session_hour',  F.hour('plug_in_ts'))
    .withColumn('estimated_cost_aud',
        F.when(F.col('tariff_price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('tariff_price_per_kwh') * F.col('energy_kwh')
             + F.col('tariff_price_per_min') * F.col('duration_min')).cast('decimal(10,2)')
        )
    )
    .withColumn('grid_cost_aud',
        F.when(F.col('grid_price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('grid_price_per_kwh') * F.col('energy_kwh')).cast('decimal(10,2)')
        )
    )
    .withColumn('gross_margin_aud',
        F.when(F.col('estimated_cost_aud').isNotNull() & F.col('grid_cost_aud').isNotNull(),
            (F.col('estimated_cost_aud') - F.col('grid_cost_aud')).cast('decimal(10,2)')
        )
    )
    .withColumn('power_efficiency_pct',
        F.when(
            F.col('charger_power_kw').isNotNull() & (F.col('charger_power_kw') > 0)
            & F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
            (F.col('energy_kwh') / (F.col('charger_power_kw') * F.col('duration_min') / 60) * 100
            ).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'realtime_sessions_enriched: {realtime_enriched.count()} rows')

In [ ]:
# Cell 8 — Build invoice_summary
invoice_summary = (
    invoices
    .groupBy('invoice_year', 'invoice_month')
    .agg(
        F.count('invoice_id').alias('total_invoices'),
        F.sum('file_size_kb').cast('decimal(12,2)').alias('total_file_size_kb'),
        F.avg('file_size_kb').cast('decimal(8,2)').alias('avg_file_size_kb'),
        F.min('invoice_date').alias('earliest_invoice_date'),
        F.max('invoice_date').alias('latest_invoice_date'),
        F.min('invoice_number').alias('invoice_number_min'),
        F.max('invoice_number').alias('invoice_number_max'),
    )
    .withColumn(
        'report_period',
        F.concat_ws('', F.col('invoice_year').cast('string'),
                    F.lpad(F.col('invoice_month').cast('string'), 2, '0'))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'invoice_summary: {invoice_summary.count()} rows')

In [ ]:
# Cell 9 — Build kpi_sla_dashboard
kpi_sla_dashboard = (
    kpi_report.alias('kpi')
    .join(sla_report.alias('sla'), on='report_period', how='inner')
    .select(
        F.col('kpi.report_period'),
        F.col('kpi.report_year'),
        F.col('kpi.report_month'),
        F.col('kpi.generated_at').alias('kpi_generated_at'),
        'kpi.total_sessions',
        'kpi.total_energy_kwh',
        'kpi.total_revenue_aud',
        'kpi.avg_session_duration_min',
        'kpi.avg_energy_per_session_kwh',
        'sla.uptime_pct',
        'sla.avg_response_time_sec',
        'sla.incidents',
        'sla.mttr_hours',
        'sla.chargers_meeting_sla',
        'sla.total_chargers',
        F.when(F.col('sla.total_chargers') > 0,
            (F.col('sla.chargers_meeting_sla') / F.col('sla.total_chargers') * 100
            ).cast('decimal(6,2)')
        ).alias('sla_compliance_pct'),
        F.when(F.col('kpi.total_sessions') > 0,
            (F.col('kpi.total_revenue_aud') / F.col('kpi.total_sessions')).cast('decimal(8,2)')
        ).alias('revenue_per_session_aud'),
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'kpi_sla_dashboard: {kpi_sla_dashboard.count()} rows')

In [ ]:
# Cell 10 — Build state_performance
state_performance = (
    state_bdown
    .join(station_counts, on='state_code', how='left')
    .join(weather_state,  on='state_code', how='left')
    .join(state_names,    on='state_code', how='left')
    .withColumn(
        'revenue_per_charger_aud',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.col('revenue_aud') / F.col('total_chargers')).cast('decimal(10,2)')
        )
    )
    .withColumn(
        'sessions_per_charger',
        F.when(F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            (F.col('sessions') / F.col('total_chargers')).cast('decimal(8,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_load_type',   F.lit(LOAD_TYPE))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
)

print(f'state_performance: {state_performance.count()} rows')

In [ ]:
# Cell 11 — Write all Gold marts
mart_configs = [
    {
        'name'          : 'realtime_sessions_enriched',
        'df'            : realtime_enriched,
        'merge_keys'    : ['session_id'],
        'partition_cols': ['session_year', 'session_month'],
    },
    {
        'name'          : 'invoice_summary',
        'df'            : invoice_summary,
        'merge_keys'    : ['invoice_year', 'invoice_month'],
        'partition_cols': ['invoice_year', 'invoice_month'],
    },
    {
        'name'          : 'kpi_sla_dashboard',
        'df'            : kpi_sla_dashboard,
        'merge_keys'    : ['report_period'],
        'partition_cols': ['report_year', 'report_month'],
    },
    {
        'name'          : 'state_performance',
        'df'            : state_performance,
        'merge_keys'    : ['state_code', 'report_period'],
        'partition_cols': ['report_year', 'report_month'],
    },
]

results = []
for mc in mart_configs:
    print(f"  Writing {mc['name']} ...", end=' ')
    mode = write_gold(mc['df'], mc['name'], mc['merge_keys'], mc['partition_cols'])
    rows = mc['df'].count()
    results.append({'mart': mc['name'], 'rows': rows, 'mode': mode})
    print(f"{mode} ({rows} rows)")

print('\nAll Gold blob marts written.')

In [ ]:
# Cell 12 — Run summary
print('=' * 75)
print('GOLD BLOB ALL MARTS v3 — RUN SUMMARY')
print('=' * 75)
print(f'  load_type : {LOAD_TYPE}')
print(f'  run_date  : {RUN_DATE}  (realtime)')
print(f'  run_year  : {RUN_YEAR}  run_month: {RUN_MONTH}  (invoices + reports)')
print(f'  run_ts    : {RUN_TS}')
print()
print(f"  {'MART':<35} {'MODE':<10} {'ROWS':>8}")
print(f"  {'-'*35} {'-'*10} {'-'*8}")
for r in results:
    print(f"  {r['mart']:<35} {r['mode']:<10} {r['rows']:>8}")

print()
print('Gold blob transformation complete.')

In [ ]:
# Cell 13 — Verify Gold output
print('GOLD BLOB VERIFICATION')
print('-' * 55)
for mc in mart_configs:
    path = f"{GOLD_BLOB}/{mc['name']}"
    try:
        df = spark.read.format('delta').load(path)
        print(f"  {mc['name']:<35} rows={df.count():>8}  cols={len(df.columns)}")
    except Exception as e:
        print(f"  {mc['name']:<35} ERROR: {e}")

print('\nKPI + SLA Dashboard:')
spark.read.format('delta').load(f'{GOLD_BLOB}/kpi_sla_dashboard') \
    .select('report_period', 'total_sessions', 'total_revenue_aud',
            'uptime_pct', 'sla_compliance_pct') \
    .orderBy('report_period').show(truncate=False)